# The pipeline — reviews bronze → silver → gold, incrementally

Lakeflow Spark Declarative Pipeline. The session's title, literally: a pipeline. The medallion flow as one runnable graph:

| Layer | Dataset | Type | AI calls per refresh |
| --- | --- | --- | --- |
| bronze | `bronze_delivery_reviews_st` | STREAMING TABLE (Auto Loader) | — |
| silver | `silver_delivery_reviews_masked_st` | STREAMING TABLE + expectations | `ai_mask`, once per **new** row |
| silver | `silver_delivery_reviews_enriched_st` | STREAMING TABLE | 6 functions, once per **new** row |
| gold | `gold_delivery_issue_trend_mv` | MATERIALIZED VIEW | 0 |
| gold | `gold_support_escalation_queue_st` | STREAMING TABLE + flow | `ai_gen`, once per **new** negative review |
| gold | `gold_ai_quality_mv` | MATERIALIZED VIEW | 0 |

**Incremental semantics:** streaming tables process only **new** rows each refresh; materialized views recompute cheap aggregations over what's already stored. **Triggered mode:** nothing runs between refreshes — zero idle cost.

**Demo flow:** refresh 1 → 25 rows. Land batch 2 (notebook 03, last cell). Refresh 2 → only 15 new rows flow through, and the AI step pays for only those.

Dataset names end in `_st` / `_mv` so they coexist with notebook 02's tables in the same schema (`workspace.food_delivery_demo`).

**Source of truth:** `pipeline/food_delivery_reviews_pipeline.sql` — attached to the `food_delivery_reviews_pipeline` pipeline as its library. (Pipeline source files can't be notebook cells with `%sql` magic — pipelines ignore those cells — so the SQL lives in a clean `.sql` file; this notebook is the walkthrough & runbook.)

In [0]:
%sql
-- Run after each refresh to watch the pipeline grow incrementally.
-- Refresh 1: bronze 25, masked 24, enriched 24, queue 15.
-- Refresh 2: bronze 40, masked 39, enriched 39, queue 25 — AI only paid for the new rows.
SELECT 'bronze_st'                AS dataset, count(*) AS rows FROM workspace.food_delivery_demo.bronze_delivery_reviews_st
UNION ALL SELECT 'masked_st',         count(*) FROM workspace.food_delivery_demo.silver_delivery_reviews_masked_st
UNION ALL SELECT 'enriched_st',       count(*) FROM workspace.food_delivery_demo.silver_delivery_reviews_enriched_st
UNION ALL SELECT 'escalation_queue',  count(*) FROM workspace.food_delivery_demo.gold_support_escalation_queue_st
UNION ALL SELECT 'issue_trend_groups',count(*) FROM workspace.food_delivery_demo.gold_delivery_issue_trend_mv;

## Running the demo

1. **Refresh 1** (Start): ingests `batch_1_reviews.csv` (25 rows) → masked 24 (junk dropped by the `too_short` expectation) → enriched 24 → golds built.
2. **Land batch 2**: run the last cell of notebook `03_incremental_data_setup` (or drag `batch_2_reviews.csv` into the volume in Catalog Explorer).
3. **Refresh 2**: bronze +15, masked +14 (the `meh` row dropped), enriched +14 — the AI step only pays for the new rows, never re-runs the first 24.
4. Show in the UI: `source_file` in bronze (which file each row came from), the **Data quality** tab (expectation drop/warn metrics), the flow graph, and `gold_ai_quality_mv`.

**Known trade-off (teaching point):** row-level expectations cannot see already-ingested history, so the cross-batch duplicate (review 35 = copy of review 8) is enriched once more but is deduplicated in gold via `row_number` over `text_hash`.